## Clone Ultralytics

In [1]:
# ✅ Step 1: Clone ultralytics source locally
!git clone https://github.com/ultralytics/ultralytics /kaggle/working/yolov8-patched

Cloning into '/kaggle/working/yolov8-patched'...
remote: Enumerating objects: 61709, done.
remote: Counting objects: 100% (591/591), done.
remote: Compressing objects: 100% (327/327), done.
remote: Total 61709 (delta 470), reused 264 (delta 264), pack-reused 61118 (from 4)
Receiving objects: 100% (61709/61709), 33.41 MiB | 24.74 MiB/s, done.
Resolving deltas: 100% (45841/45841), done.


## Patch tasks.py for PyTorch 2.6+ Compatibility

In [2]:
# ✅ Step 2: Patch torch.load in tasks.py
import re

tasks_path = "/kaggle/working/yolov8-patched/ultralytics/nn/tasks.py"

# Replace the strict torch.load with weights_only=False version
with open(tasks_path, "r") as f:
    content = f.read()

# Patch both occurrences (there are 2 load calls)
patched = re.sub(
    r'torch\.load\(([^,]+), map_location=([^\)]+)\)',
    r'torch.load(\1, map_location=\2, weights_only=False)',
    content
)

with open(tasks_path, "w") as f:
    f.write(patched)

print("✅ tasks.py patched with weights_only=False.")

✅ tasks.py patched with weights_only=False.


## Install Patched YOLOv8 in Editable Mode

In [3]:
# ✅ Step 3: Install the patched YOLOv8 package
%cd /kaggle/working/yolov8-patched
!pip install -e . --quiet

/kaggle/working/yolov8-patched
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.3 MB/s eta 0:00:00:00:0100:01
  Building editable for ultralytics (pyproject.toml) ... done


## Confirm Patch Works and Load Model with GPU

In [4]:
from ultralytics import YOLO
import torch

print("✅ PyTorch version:", torch.__version__)
print("✅ Ultralytics version:", YOLO.__module__)

# ✅ Load model using patched YOLOv8
model = YOLO("yolov8n.pt")  # should now load without unpickling error


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ PyTorch version: 2.6.0+cu124
✅ Ultralytics version: ultralytics.models.yolo.model


100%|██████████| 6.25M/6.25M [00:00<00:00, 78.0MB/s]


## Define Paths and Create Folder Structure

In [8]:
import os

BASE_DIR = "/kaggle/input/helmet-detection"
IMAGES_DIR = f"{BASE_DIR}/images"
ANNOTATIONS_DIR = f"{BASE_DIR}/annotations"

OUTPUT_DIR = "/kaggle/working/helmet-dataset"
os.makedirs(f"{OUTPUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/val", exist_ok=True)


## Convert VOC XML → YOLO Format

In [9]:
import xml.etree.ElementTree as ET
import random
import shutil
from tqdm import tqdm

class_map = {
    "With Helmet": 0,
    "Without Helmet": 1
}

def convert_to_yolo_bbox(bbox, img_width, img_height):
    xmin, ymin, xmax, ymax = bbox
    x_center = (xmin + xmax) / 2 / img_width
    y_center = (ymin + ymax) / 2 / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height
    return x_center, y_center, width, height

image_files = sorted(os.listdir(IMAGES_DIR))
random.seed(42)
random.shuffle(image_files)

split_idx = int(0.8 * len(image_files))
train_imgs = image_files[:split_idx]
val_imgs = image_files[split_idx:]

def process_split(split_name, image_list):
    for img_name in tqdm(image_list, desc=f"Processing {split_name}"):
        xml_file = os.path.join(ANNOTATIONS_DIR, img_name.replace(".png", ".xml"))
        img_file = os.path.join(IMAGES_DIR, img_name)
        label_file = os.path.join(OUTPUT_DIR, f"labels/{split_name}", img_name.replace(".png", ".txt"))
        out_img_file = os.path.join(OUTPUT_DIR, f"images/{split_name}", img_name)

        # Copy image
        shutil.copy2(img_file, out_img_file)

        # Parse annotation
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find("size")
        w, h = int(size.find("width").text), int(size.find("height").text)

        with open(label_file, "w") as f:
            for obj in root.findall("object"):
                class_name = obj.find("name").text
                if class_name not in class_map:
                    continue
                cls_id = class_map[class_name]
                bndbox = obj.find("bndbox")
                xmin = int(float(bndbox.find("xmin").text))
                ymin = int(float(bndbox.find("ymin").text))
                xmax = int(float(bndbox.find("xmax").text))
                ymax = int(float(bndbox.find("ymax").text))

                x_center, y_center, width, height = convert_to_yolo_bbox((xmin, ymin, xmax, ymax), w, h)
                f.write(f"{cls_id} {x_center} {y_center} {width} {height}\n")

process_split("train", train_imgs)
process_split("val", val_imgs)


Processing val: 100%|██████████| 153/153 [00:04<00:00, 34.15it/s]


## Write helmet.yaml Config File

In [10]:
yaml_content = f"""
path: {OUTPUT_DIR}
train: images/train
val: images/val

names:
  0: With Helmet
  1: Without Helmet
"""

with open("helmet.yaml", "w") as f:
    f.write(yaml_content)

!cat helmet.yaml



path: /kaggle/working/helmet-dataset
train: images/train
val: images/val

names:
  0: With Helmet
  1: Without Helmet


## Train the YOLOv8n Model (Using Patched Ultralytics)

In [11]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

# Train
model.train(
    data="helmet.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="helmet_yolov8n"
)


Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=helmet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=helmet_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, p

100%|██████████| 755k/755k [00:00<00:00, 16.4MB/s]


Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

100%|██████████| 5.35M/5.35M [00:00<00:00, 68.3MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2332.1±1201.8 MB/s, size: 610.5 KB)


...

                   all        151        285      0.727        0.8      0.789      0.474



50 epochs completed in 0.162 hours.
Optimizer stripped from /kaggle/working/yolov8-patched/runs/detect/helmet_yolov8n/weights/last.pt, 6.2MB
Optimizer stripped from /kaggle/working/yolov8-patched/runs/detect/helmet_yolov8n/weights/best.pt, 6.2MB

Validating /kaggle/working/yolov8-patched/runs/detect/helmet_yolov8n/weights/best.pt...
Ultralytics 8.3.154 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  2.08it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        151        285      0.782      0.752      0.795      0.479
           With Helmet        109        193      0.826      0.798      0.872      0.575
        Without Helmet         53         92      0.739      0.707      0.719      0.382
Speed: 0.2ms preprocess, 2.4ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /kaggle/working/yolov8-patched/runs/detect/helmet_yolov8n


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7969e1797390>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

## Zip the Best Weights for Download

In [14]:
# ✅ Download official YOLOv8n config file (architecture)
!wget -q https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/models/v8/yolov8n.yaml

# ✅ Save state_dict only (safe for Hugging Face / CPU inference)
import torch
torch.save(model.model.state_dict(), "best_state_dict.pt")

# ✅ Zip both files for download or deployment
!zip -j model_export.zip best_state_dict.pt yolov8n.yaml

	zip warning: name not matched: yolov8n.yaml
updating: best_state_dict.pt (deflated 41%)


In [15]:
!ls -lh

total 31M
-rw-r--r--  1 root root  12M Jun 14 00:51 best_state_dict.pt
-rw-r--r--  1 root root  764 Jun 14 00:14 CITATION.cff
-rw-r--r--  1 root root  18K Jun 14 00:14 CONTRIBUTING.md
drwxr-xr-x  2 root root 4.0K Jun 14 00:14 docker
drwxr-xr-x  4 root root 4.0K Jun 14 00:14 docs
drwxr-xr-x 19 root root 4.0K Jun 14 00:14 examples
-rw-r--r--  1 root root  119 Jun 14 00:24 helmet.yaml
-rw-r--r--  1 root root  34K Jun 14 00:14 LICENSE
-rw-r--r--  1 root root  31K Jun 14 00:14 mkdocs.yml
-rw-r--r--  1 root root 6.9M Jun 14 00:51 model_export.zip
-rw-r--r--  1 root root 7.6K Jun 14 00:14 pyproject.toml
-rw-r--r--  1 root root  33K Jun 14 00:14 README.md
-rw-r--r--  1 root root  33K Jun 14 00:14 README.zh-CN.md
drwxr-xr-x  3 root root 4.0K Jun 14 00:34 runs
drwxr-xr-x  2 root root 4.0K Jun 14 00:14 tests
drwxr-xr-x 13 root root 4.0K Jun 14 00:18 ultralytics
drwxr-xr-x  2 root root 4.0K Jun 14 00:15 ultralytics.egg-info
-rw-r--r--  1 root root 5.4M Jun 14 00:34 yolo11n.pt
-rw-r--r--  1 root ro

In [17]:
!wget -q https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/models/v8/yolov8n.yaml


In [18]:
!zip -j model_export.zip best_state_dict.pt yolov8n.yaml


	zip warning: name not matched: yolov8n.yaml
updating: best_state_dict.pt (deflated 41%)


In [25]:
!find /kaggle/working -name "yolov8n.yaml"

In [28]:
!wget -O yolov8n.yaml https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v8/yolov8n.yaml

--2025-06-14 01:01:19--  https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v8/yolov8n.yaml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-06-14 01:01:19 ERROR 404: Not Found.

